## **Librerias y recursos**

EXP3 con ROC-AUC y PR

In [1]:
!lscpu |grep 'Model name'
!nvidia-smi -L

Model name:                           Intel(R) Xeon(R) CPU @ 2.00GHz
GPU 0: Tesla P100-PCIE-16GB (UUID: GPU-c4f5caca-6105-27bc-3e89-67b67d37612d)


In [2]:
!pip install -q eva-decord==0.6.1
!pip install -q --no-deps lightning==2.4.0 torchmetrics==1.4.0
!pip install -q moviepy==1.0.3
!pip install -q av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 101.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 19.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.8/868.8 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 MB 44.9 MB/s eta 0:00:00:00:0100:01


In [3]:
import os, glob, random, time, csv, math, json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import lightning as L
from torchmetrics.classification import MulticlassAccuracy, MulticlassF1Score

import torchvision
from decord import VideoReader, cpu, gpu
import decord
decord.bridge.set_bridge('torch')

# **Hiperparámetros**

In [4]:
!ls '/kaggle/input/crimeucfdataset/Anomaly_Dataset/Anomaly_Videos/'

Anomaly-Videos-Part-1  ReadMe-Anomaly-Detection.txt
Anomaly-Videos-Part-2  Temporal_Anomaly_Annotation_for_Testing_Videos.txt
Normal-Videos-Part-1


In [5]:
DATA_DIR = '/kaggle/input/crimeucfdataset/Anomaly_Dataset/Anomaly_Videos/'
CLASS_PATHS = {
    "Normal":   Path(DATA_DIR, "Normal-Videos-Part-1"),
    "Abuse":    Path(DATA_DIR, "Anomaly-Videos-Part-1", "Abuse"),
    "Fighting": Path(DATA_DIR, "Anomaly-Videos-Part-2", "Fighting"),
}
CLASSES = list(CLASS_PATHS.keys())
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IMG_SIZE  = 112           # r3d_18 esta preentrenado a 112x112
CLIP_LEN  = 16            # frames por clip
FPS_READ  = None          # (opcional) para fijar fps, dejar None para muestreo uniforme por índice
HOP_TRAIN = CLIP_LEN//2  #CLIP_LEN//2   # salto entre clips en train (si uso multi-samples)
HOP_VAL   = CLIP_LEN//2
TRAIN_SAMPLES_PER_VIDEO = 16 # exp3
VAL_SAMPLES_PER_VIDEO   = 20 # se deja este valor para tener una cantidad decente de clips en validación
VAL_RATIO = 0.2
BATCH_SIZE  = 4
NUM_WORKERS = 4
MAX_EPOCHS  = 20
LR = 5e-5 #1e-4 #2e-4 exp1
WD = 0.01 #0.01 exp1
RATE = 2 # muestreo temporal cada 2 frames
PRECISION = "bf16-mixed" if torch.cuda.is_available() else 32

USE_AUGS = True
HARD_NEG_THRESHOLD = 0.015
HARD_NEG_TRIES = 3
MAJORITY_NAME = 'Normal'
KEEP_RATIO = 0.84 
MAX_KEEP = 90 #reduzco 25% de la clase mayoritaria
MINORITY_NAMES = ['Abuse', 'Fighting']
OVERSAMPLE_FACTS = {'Abuse': 1.7, 'Fighting': 1.6} # aumento 5% de cada minoría

KINETICS_MEAN = torch.tensor([0.43216, 0.394666, 0.37645]).view(3,1,1,1)
KINETICS_STD = torch.tensor([0.22803, 0.22145, 0.216989]).view(3,1,1,1)
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SHUFFLE_SEED = 123

In [6]:
# === EXPERIMENT TAG ===
EXP_TAG = "EXP3"

# Directorios estándar (coherentes con tu esquema de CM)
CONF_MAT_DIR = f"CONF_MATRICES_{EXP_TAG}"
ROC_DIR      = f"ROC_CURVES_{EXP_TAG}"
PR_DIR       = f"PR_CURVES_{EXP_TAG}"

os.makedirs(CONF_MAT_DIR, exist_ok=True)
os.makedirs(ROC_DIR, exist_ok=True)
os.makedirs(PR_DIR, exist_ok=True)

# **PREPROCESAMIENTO DE DATOS**

In [7]:
def list_videos_from_dict(class_paths):
    """Returns list of (path, class_index) from a dict {class_name: Path}."""
    items = []
    for ci, cname in enumerate(class_paths.keys()):
        base = class_paths[cname]
        mp4s = sorted(glob.glob(str(base / "*.mp4")))
        avis = sorted(glob.glob(str(base / "*.avi")))
        files = mp4s + avis
        items += [(f, ci) for f in files]
    random.shuffle(items)
    return items

In [8]:
def stratified_split(items,min_val_per_class=1):
    """Stratified split at video (path) level. Returns (train_items, val_items). items: list[(path, class_idx)]."""
    rng = random.Random(SEED)
    per_class = defaultdict(list)
    for p, ci in items:
        per_class[ci].append(p)
    train_items = []
    val_items = []
    for ci, vids in per_class.items():
        vids = list(vids)
        rng.shuffle(vids)
        k = max(min_val_per_class, int(round(len(vids) * VAL_RATIO)))
        val_v = vids[:k]
        train_v = vids[k:]
        val_items += [(p, ci) for p in val_v]
        train_items += [(p, ci) for p in train_v]
    rng.shuffle(train_items)
    rng.shuffle(val_items)
    return train_items, val_items

In [9]:
import random
from collections import defaultdict

def downsample_majority(train_items, max_keep_custom=None):
    """Downsample majority class (by name) in train_items and return new list."""
    maj_id = CLASSES.index(MAJORITY_NAME)
    by_class = defaultdict(list)
    for p, ci in train_items:
        by_class[ci].append((p, ci))
    n_normal = len(by_class[maj_id])

    #esto es por si queremos bajar en validation tambien
    if max_keep_custom is not None:
        keep_n = min(int(max_keep_custom), n_normal)
    else:
        if MAX_KEEP is None:
            keep_n = int(round(n_normal * float(KEEP_RATIO)))
        else:
            keep_n = min(int(MAX_KEEP), n_normal)
    rng = random.Random(SEED)
    normal_kept = rng.sample(by_class[maj_id], keep_n) if n_normal > keep_n else by_class[maj_id]
    new_train = []
    for ci, items in by_class.items():
        if ci == maj_id:
            new_train.extend(normal_kept)
        else:
            new_train.extend(items)
            
    rng.shuffle(new_train)
    return new_train

## Heuristica para descartar muestras irrelevantes en los videos de Abuse y Fighting

In [10]:
# === Heurística de SKIP configurable por clase ===
import numpy as np, random
from pathlib import Path
from decord import VideoReader, cpu

# Clases a las que les aplicaremos modo "mix" (o "auto"); el resto -> "fixed=0"
TARGET_AUTO_CLASSES = {"Abuse", "Fighting"}

# Config global
SKIP_GLOBAL_MODE_FOR_TARGETS = "mix"   # "auto" | "fixed" | "percent" | "mix"
SKIP_FIXED_S_BY_CLASS = {              # parte fija (también usada por "mix")
    "Abuse": 6.0,
    "Fighting": 6.0,
}
SKIP_PERCENT = 0.20                    # si usas "percent"
AUTO_SAMPLE_EVERY_S = 0.5
AUTO_TH = 0.15
AUTO_CONSEC = 2
AUTO_FALLBACK_S = 3.0
MAX_SKIP_RATIO = 0.5                   # no saltar > 50% del video

# caches: path-> (skip_frames, skip_s, auto_found_bool)
_skip_cache = {}

def _estimate_fps(vr):
    try:
        return float(vr.get_avg_fps())
    except Exception:
        return 25.0

def _content_change_score_rgb(prev, cur):
    # prev/cur: ndarray (H,W,3) uint8
    return float(np.mean(np.abs(cur.astype(np.int16) - prev.astype(np.int16))) / 255.0)

# --- helpers robustos para frames de Decord (NDArray o torch.Tensor) ---
def _frame_to_numpy(frm):
    # Decord NDArray
    if hasattr(frm, "asnumpy"):
        return frm.asnumpy()
    # torch.Tensor (H, W, C) típico del bridge 'torch'
    try:
        import torch
        if isinstance(frm, torch.Tensor):
            return frm.detach().cpu().numpy()
    except Exception:
        pass
    # ya es numpy o convertible
    import numpy as np
    if isinstance(frm, np.ndarray):
        return frm
    return np.array(frm)

def _auto_find_onset_seconds(path, sample_every_s=0.5, th=0.15, consec=2):
    from decord import VideoReader, cpu
    import numpy as np

    try:
        vr = VideoReader(path, ctx=cpu(0))
    except Exception:
        return 0.0, False

    fps = _estimate_fps(vr)
    T = len(vr)
    if T < 2:
        return 0.0, False

    step = max(1, int(round(sample_every_s * fps)))
    idxs = list(range(0, T, step))
    if idxs[-1] != T - 1:
        idxs.append(T - 1)

    prev = None
    hits = 0
    for i in range(1, len(idxs)):
        cur_idx = idxs[i]
        frm = vr[cur_idx]                 # puede ser NDArray o torch.Tensor
        frm = _frame_to_numpy(frm)        # <-- normaliza a numpy
        frm_small = frm[::4, ::4, :]      # downscale rápido

        if prev is not None:
            sc = _content_change_score_rgb(prev, frm_small)
            if sc >= th:
                hits += 1
                if hits >= consec:
                    onset_idx = idxs[max(1, i - consec + 1)]
                    return onset_idx / fps, True
            else:
                hits = 0
        prev = frm_small

    return 0.0, False

def compute_skip_for_video(path, label_int, class_names):
    """
    Devuelve (skip_frames, skip_seconds, auto_found)
    - Modo por clase:
        * Si nombre de clase ∈ TARGET_AUTO_CLASSES => usa SKIP_GLOBAL_MODE_FOR_TARGETS
        * Si no => 'fixed' con 0s (o lo definido en SKIP_FIXED_S_BY_CLASS si existe)
    """
    path = str(path)
    key = (path, int(label_int))
    if key in _skip_cache:
        return _skip_cache[key]

    # nombre de clase
    if isinstance(class_names, (list, tuple)):
        cname = class_names[label_int]
    else:
        # dict nombre->id
        cname = [k for k, v in class_names.items() if v == label_int][0]

    # decidir modo local
    is_target = cname in TARGET_AUTO_CLASSES
    mode = SKIP_GLOBAL_MODE_FOR_TARGETS if is_target else "fixed"

    skip_s = 0.0
    auto_found = False

    # fijo (por clase)
    if mode in ("fixed", "mix"):
        skip_s = SKIP_FIXED_S_BY_CLASS.get(cname, 0.0)

    # percent
    if mode == "percent":
        try:
            vr = VideoReader(path, ctx=cpu(0))
            fps = _estimate_fps(vr)
            dur_s = len(vr) / fps
            skip_s = dur_s * SKIP_PERCENT
        except Exception:
            skip_s = 0.0

    # auto / mix
    if mode in ("auto", "mix"):
        auto_s, auto_found = _auto_find_onset_seconds(
            path, AUTO_SAMPLE_EVERY_S, AUTO_TH, AUTO_CONSEC
        )
        if mode == "auto":
            skip_s = auto_s if auto_s > 0 else AUTO_FALLBACK_S
        else:
            skip_s = max(skip_s, (auto_s if auto_s > 0 else AUTO_FALLBACK_S))

    # limitar por longitud del video
    try:
        vr = VideoReader(path, ctx=cpu(0))
        fps = _estimate_fps(vr)
        max_skip_s = (len(vr) / fps) * MAX_SKIP_RATIO
        skip_s = max(0.0, min(skip_s, max_skip_s))
        skip_frames = int(round(skip_s * fps))
    except Exception:
        fps = 25.0
        skip_frames = int(round(skip_s * fps))

    _skip_cache[key] = (skip_frames, skip_s, auto_found)
    return _skip_cache[key]

# === Utilidad de logs por clase para los SKIPs (sobre lista de videos base) ===
def summarize_skip_stats(items, class_to_idx=CLASS_TO_IDX, class_names=CLASSES, title="Split"):
    """
    items: lista de (path,label) de VIDEOS (no clips)
    imprime:
      - % de vídeos target con auto_onset detectado
      - media de segundos saltados por clase
    """
    import math
    from collections import defaultdict

    by_class_s = defaultdict(list)
    by_class_found = defaultdict(int)
    by_class_total = defaultdict(int)

    for path, y in items:
        sf, ss, found = compute_skip_for_video(path, int(y), class_names)
        by_class_s[int(y)].append(ss)
        by_class_total[int(y)] += 1
        if found:
            by_class_found[int(y)] += 1

    print(f"\n=== Heurística SKIP — {title} ===")
    print(f"{'Clase':15s} | {'avg skip (s)':>12s} | {'auto found %':>12s} | {'n videos':>9s}")
    print("-"*56)
    for cname, cid in class_to_idx.items():
        n = by_class_total.get(cid, 0)
        avg = (sum(by_class_s.get(cid, [])) / n) if n > 0 else 0.0
        pct = (by_class_found.get(cid, 0) / n * 100.0) if n > 0 else 0.0
        print(f"{cname:15s} | {avg:12.3f} | {pct:12.1f}% | {n:9d}")


In [11]:
import torch, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np, random
from decord import VideoReader, cpu

class VideoClipDataset(Dataset):
    """
    Soporta:
      - base_samples_per_video (train) / val_samples_per_video (val)
      - extras aug-only por clase (si ya lo tenías, mantén tu versión)
      - heurística de SKIP por video (esta celda integra el offset)
    """
    def __init__(
        self,
        items,
        clip_len=16,
        img_size=112,
        train=True,
        use_augs=True,
        base_samples_per_video=1,
        val_samples_per_video=1,
        hop_train=None,
        hop_val=None,
        fps_read=None,
        # --- si usas extras por clase, deja tus params aquí (opcional) ---
        extra_aug_class_names=None,
        extra_aug_fraction=0.0,
        extra_aug_fraction_map=None,
        class_to_idx=None,
        extra_aug_strength=0.35
    ):
        self.clip_len = int(clip_len)
        self.img_size = int(img_size)
        self.train = bool(train)
        self.use_augs = bool(use_augs)
        self.base_samples_per_video = int(base_samples_per_video)
        self.val_samples_per_video  = int(val_samples_per_video)
        self.hop_train = hop_train if hop_train is not None else self.clip_len // 2
        self.hop_val   = hop_val   if hop_val   is not None else self.clip_len // 2
        self.fps_read  = fps_read

        self.class_to_idx = class_to_idx if class_to_idx is not None else CLASS_TO_IDX

        # normaliza items -> (path,label)
        self.items = [(it[0], int(it[1])) for it in items]
        self._n_videos = len(self.items)

        # índice plano (vid_idx, aug_only)
        self._index = []
        if self.train:
            # base
            for vid_idx in range(self._n_videos):
                for _ in range(self.base_samples_per_video):
                    self._index.append((vid_idx, False))
            # extras aug-only (si se configuran)
            target_ids = set()
            if extra_aug_class_names:
                for cname in extra_aug_class_names:
                    if cname in self.class_to_idx:
                        target_ids.add(self.class_to_idx[cname])
            frac_map = {}
            if extra_aug_fraction_map:
                for cname, frac in extra_aug_fraction_map.items():
                    if cname in self.class_to_idx:
                        frac_map[self.class_to_idx[cname]] = float(frac)
            for vid_idx, (_, y) in enumerate(self.items):
                if y in target_ids:
                    frac = frac_map.get(y, float(extra_aug_fraction))
                    extra = max(0, int(round(self.base_samples_per_video * frac)))
                    for _ in range(extra):
                        self._index.append((vid_idx, True))
        else:
            for vid_idx in range(self._n_videos):
                for _ in range(self.val_samples_per_video):
                    self._index.append((vid_idx, False))

        random.shuffle(self._index)

    def __len__(self):
        return len(self._index)

    def _choose_indices_with_hop(self, T, k, hop, start_offset=0, jitter=False):
        L = self.clip_len
        if T <= L + start_offset:
            start = max(start_offset, T - L)
            return list(range(start, start + L))
        starts = list(range(start_offset, max(start_offset+1, T - L + 1), max(1, hop))) or [start_offset]
        start = starts[min(k % len(starts), len(starts) - 1)]
        if jitter and len(starts) > 1:
            start = int(np.clip(start + random.randint(-hop//2, hop//2), start_offset, T - L))
        return list(range(start, start + L))

    def _apply_aug(self, x, strength=0.20):
        if random.random() > 0.5:
            x = torch.flip(x, dims=[3])
        delta = (random.random() - 0.5) * strength
        scale = 1.0 + (random.random() - 0.5) * (2*strength)
        return torch.clamp(x * scale + delta, 0.0, 1.0)

    def __getitem__(self, idx):
        vid_idx, aug_only = self._index[idx]
        path, label = self.items[vid_idx]

        # === NEW: obtener offset de inicio por video (según heurística y clase) ===
        skip_frames, skip_s, auto_found = compute_skip_for_video(path, int(label), CLASSES)

        try:
            vr = VideoReader(path, ctx=cpu(0))
            T = len(vr)
            idxs = self._choose_indices_with_hop(
                T=T,
                k=idx,
                hop=(self.hop_train if self.train else self.hop_val),
                start_offset=skip_frames,
                jitter=self.train  # sin jitter en val
            )
            frames = vr.get_batch(idxs).permute(3,0,1,2).float() / 255.0
        except Exception:
            frames = torch.zeros(3, self.clip_len, max(128, self.img_size), max(128, self.img_size), dtype=torch.float32)

        frames = F.interpolate(
            frames.unsqueeze(0),
            size=(self.clip_len, self.img_size, self.img_size),
            mode="trilinear",
            align_corners=False
        ).squeeze(0)

        # augs
        if self.train:
            if aug_only:
                frames = self._apply_aug(frames, strength=0.35)
            elif self.use_augs:
                frames = self._apply_aug(frames, strength=0.20)

        return frames, label

print("[OK] VideoClipDataset con heurística SKIP por clase y sin jitter en val.")

[OK] VideoClipDataset con heurística SKIP por clase y sin jitter en val.


In [12]:
def collate_and_normalize(batch):
    frames, labels = zip(*batch)
    X = torch.stack(frames, dim=0)          # (B,C,T,H,W)
    y = torch.tensor(labels, dtype=torch.long)
    return X, y

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction="mean", label_smoothing=0.0):
        super().__init__()
        self.gamma = float(gamma)
        self.reduction = reduction
        self.label_smoothing = float(label_smoothing)

        # alpha puede ser None, list, numpy o tensor
        if alpha is not None:
            alpha_t = torch.as_tensor(alpha, dtype=torch.float32)
            # registrar como buffer -> se mueve con .to(device)
            self.register_buffer("alpha", alpha_t)
        else:
            self.alpha = None

    def forward(self, logits, targets):
        """
        logits: [B, C]  (en cuda o cpu)
        targets: [B]    (mismo device que logits)
        """
        # asegurar mismo device/dtype para 'alpha'
        weight = None
        if self.alpha is not None:
            weight = self.alpha.to(logits.device, dtype=logits.dtype)

        ce = F.cross_entropy(
            logits, targets, weight=weight,
            reduction="none", label_smoothing=self.label_smoothing
        )  # [B]
        pt = torch.exp(-ce)              # p_t
        focal = (1.0 - pt) ** self.gamma # [B]
        loss = focal * ce                # [B]

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss

In [14]:
def compute_alpha_cb_from_items(items, num_classes=NUM_CLASSES, class_to_idx=CLASS_TO_IDX):
    counts = np.zeros(int(num_classes), dtype=np.float32)
    for path, label in items:
        label = int(label)
        if 0 <= label < num_classes:
            counts[label] += 1
        else:
            raise ValueError(f"Etiqueta fuera de rango: {label}")
    # evitar divisiones por cero
    counts = np.clip(counts, 1, None)
    inv_freq = 1.0 / counts
    alpha = inv_freq / inv_freq.sum() * num_classes  # normaliza para que promedio sea ~1
    return torch.tensor(alpha, dtype=torch.float32)


In [15]:
# FUNCION EXTERNA PARA CREAR 
def make_loss(loss_type='ce', alpha=None, gamma=2.0, label_smoothing=0.0, reduction='mean'):
    if loss_type == 'ce':
        return nn.CrossEntropyLoss(weight=alpha, label_smoothing=label_smoothing, reduction=reduction)
    elif loss_type == 'focal':
        return FocalLoss(alpha=alpha, gamma=gamma, label_smoothing=label_smoothing, reduction=reduction)
    else: #self.criterion = FocalLoss(alpha=alpha, gamma=2.0,  label_smoothing=0.05)
        raise ValueError(f"loss_type desconocido: {loss_type}")
print("[OK] FocalLoss y make_loss listos.")

[OK] FocalLoss y make_loss listos.


In [16]:
from collections import Counter

def dataset_stats(ds, split_name="train"):
    print(f"\n=== Estadísticas de {split_name.upper()} ===")
    print(f"Total de clips (len): {len(ds):,}")

    # Contar clips por clase
    class_counts = Counter()
    aug_only_counts = Counter()

    for vid_idx, aug_only in ds._index:
        _, y = ds.items[vid_idx]
        class_counts[y] += 1
        if aug_only:
            aug_only_counts[y] += 1

    # Mostrar por clase
    total = sum(class_counts.values())
    print(f"{'Clase':15s} | {'Clips totales':>14s} | {'%':>6s} | {'Extras (aug-only)':>18s}")
    print("-" * 60)
    for cid in range(len(CLASS_TO_IDX)):
        cname = [k for k, v in CLASS_TO_IDX.items() if v == cid][0]
        n_total = class_counts.get(cid, 0)
        n_aug = aug_only_counts.get(cid, 0)
        pct = (n_total / total * 100) if total > 0 else 0
        print(f"{cname:15s} | {n_total:14d} | {pct:5.1f}% | {n_aug:18d}")
    print("-" * 60)
    print(f"Total clips: {total:,}")
    print()

# **MAIN PRE-PROCESSING FLOW**

In [17]:
all_items = list_videos_from_dict(CLASS_PATHS)
print("Total videos:", len(all_items))

train_items, val_items = stratified_split(all_items, min_val_per_class=1)
train_items = downsample_majority(train_items, max_keep_custom=MAX_KEEP)

# Reporte por clase
print(
    "Videos en Train por clase:",
    {CLASSES[i]: sum(1 for it in train_items if it[1] == i) for i in range(NUM_CLASSES)},
    "Total:", len(train_items)
)
print(
    "Videos en Val por clase:",
    {CLASSES[i]: sum(1 for it in val_items if it[1] == i) for i in range(NUM_CLASSES)},
    "Total:", len(val_items)
)

# Resumen por split de la heurística de skip (sobre VIDEOS base)
summarize_skip_stats(train_items, class_to_idx=CLASS_TO_IDX, class_names=CLASSES, title="TRAIN (videos)")
summarize_skip_stats(val_items,   class_to_idx=CLASS_TO_IDX, class_names=CLASSES, title="VAL (videos)")


# === Datasets con N clips por video por epoch ===
EXTRA_FRACTIONS = {"Abuse": 0.02, "Fighting": 0.05} # augmentations extra solo para estas clases

train_ds = VideoClipDataset(
    train_items,
    clip_len=CLIP_LEN,
    img_size=IMG_SIZE,
    train=True,
    use_augs=True,
    base_samples_per_video=TRAIN_SAMPLES_PER_VIDEO,
    val_samples_per_video=VAL_SAMPLES_PER_VIDEO,
    hop_train=HOP_TRAIN,
    hop_val=HOP_VAL,
    fps_read=FPS_READ,
    extra_aug_class_names=list(EXTRA_FRACTIONS.keys()),
    extra_aug_fraction=0.0,                 # se ignora porque pasamos el map
    extra_aug_fraction_map=EXTRA_FRACTIONS, # ← por clase
    class_to_idx=CLASS_TO_IDX,
    extra_aug_strength=0.35
)

val_ds = VideoClipDataset(
    val_items,
    clip_len=CLIP_LEN,
    img_size=IMG_SIZE,
    train=False,
    use_augs=False,
    base_samples_per_video=TRAIN_SAMPLES_PER_VIDEO,   # (ignorado en val)
    val_samples_per_video=VAL_SAMPLES_PER_VIDEO, # <-- AHORA SÍ se usa
    hop_train=HOP_TRAIN,
    hop_val=HOP_VAL,
    fps_read=FPS_READ
)

# === DataLoaders ===
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    drop_last=True,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_and_normalize,
    persistent_workers=(NUM_WORKERS > 0)
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    drop_last=False,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_and_normalize,
    persistent_workers=(NUM_WORKERS > 0)
)
print("Total batches en train_loader:", len(train_loader))
print("Total batches en val_loader:", len(val_loader))

# α class-balanced - no lo uso en este exp2
alpha_cb = compute_alpha_cb_from_items(train_items)
print("Alpha class-balanced:", [round(float(a), 4) for a in alpha_cb.tolist()])


Total videos: 250
Videos en Train por clase: {'Normal': 90, 'Abuse': 40, 'Fighting': 40} Total: 170
Videos en Val por clase: {'Normal': 30, 'Abuse': 10, 'Fighting': 10} Total: 50

=== Heurística SKIP — TRAIN (videos) ===
Clase           | avg skip (s) | auto found % |  n videos
--------------------------------------------------------
Normal          |        0.000 |          0.0% |        90
Abuse           |       14.496 |         25.0% |        40
Fighting        |       15.469 |         17.5% |        40

=== Heurística SKIP — VAL (videos) ===
Clase           | avg skip (s) | auto found % |  n videos
--------------------------------------------------------
Normal          |        0.000 |          0.0% |        30
Abuse           |        5.900 |         10.0% |        10
Fighting        |        6.000 |          0.0% |        10
Total batches en train_loader: 690
Total batches en val_loader: 250
Alpha class-balanced: [0.5455, 1.2273, 1.2273]


In [18]:
# Ejecutar para train y val
dataset_stats(train_ds, "train")
dataset_stats(val_ds, "val")


=== Estadísticas de TRAIN ===
Total de clips (len): 2,760
Clase           |  Clips totales |      % |  Extras (aug-only)
------------------------------------------------------------
Normal          |           1440 |  52.2% |                  0
Abuse           |            640 |  23.2% |                  0
Fighting        |            680 |  24.6% |                 40
------------------------------------------------------------
Total clips: 2,760


=== Estadísticas de VAL ===
Total de clips (len): 1,000
Clase           |  Clips totales |      % |  Extras (aug-only)
------------------------------------------------------------
Normal          |            600 |  60.0% |                  0
Abuse           |            200 |  20.0% |                  0
Fighting        |            200 |  20.0% |                  0
------------------------------------------------------------
Total clips: 1,000



# **ENTRENAMIENTO** #

In [19]:
import itertools
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize

def plot_confusion_matrix(cm, class_names, title="Confusion Matrix",
                          normalize=False, cmap="Blues", savepath=None):
    cm = np.array(cm, dtype=float)
    n = cm.shape[0]
    fig, ax = plt.subplots(figsize=(7,6))
    im = ax.imshow(cm, cmap=cmap)
    ax.set_xticks(np.arange(n), labels=class_names)
    ax.set_yticks(np.arange(n), labels=class_names)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i in range(n):
        for j in range(n):
            v = cm[i,j]
            color_text = "black" if cm[i,j] < cm.max()/2 else "white"
            ax.text(j, i, f"{int(round(v))}", ha="center", va="center",
                    fontsize=11, color=color_text, fontweight="bold")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, dpi=220, bbox_inches="tight")
    plt.close(fig)

def plot_roc_ovr(scores, y_true, class_names, title, savepath):
    """
    scores: np.ndarray [N, C] con probabilidades/logits normalizados por clase
    y_true: np.ndarray [N] con índices de clase verdadera
    """
    C = len(class_names)
    Y = label_binarize(y_true, classes=list(range(C)))  # [N, C]
    fig, ax = plt.subplots(figsize=(7,6))

    aucs = {}
    for i, cname in enumerate(class_names):
        fpr, tpr, _ = roc_curve(Y[:, i], scores[:, i])
        auc_i = auc(fpr, tpr)
        aucs[cname] = float(auc_i)
        ax.plot(fpr, tpr, label=f"{cname} (AUC={auc_i:.3f})")

    # Micro-average
    fpr_micro, tpr_micro, _ = roc_curve(Y.ravel(), scores.ravel())
    auc_micro = auc(fpr_micro, tpr_micro)
    ax.plot(fpr_micro, tpr_micro, linestyle="--", label=f"micro (AUC={auc_micro:.3f})")

    # Macro-average
    macro_auc = np.mean(list(aucs.values()))

    ax.plot([0,1], [0,1], color="gray", linestyle=":")
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR (Recall)")
    ax.set_title(title + f"\nMacro-AUC={macro_auc:.3f} | Micro-AUC={auc_micro:.3f}")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(savepath, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return aucs, macro_auc, auc_micro


def plot_pr_ovr(scores, y_true, class_names, title, savepath):
    C = len(class_names)
    Y = label_binarize(y_true, classes=list(range(C)))
    fig, ax = plt.subplots(figsize=(7,6))

    aps = {}
    for i, cname in enumerate(class_names):
        precision, recall, _ = precision_recall_curve(Y[:, i], scores[:, i])
        ap_i = average_precision_score(Y[:, i], scores[:, i])
        aps[cname] = float(ap_i)
        ax.plot(recall, precision, label=f"{cname} (AP={ap_i:.3f})")

    # Micro-average (apilando)
    precision_micro, recall_micro, _ = precision_recall_curve(Y.ravel(), scores.ravel())
    ap_micro = average_precision_score(Y.ravel(), scores.ravel())
    ax.plot(recall_micro, precision_micro, linestyle="--", label=f"micro (AP={ap_micro:.3f})")

    ap_macro = np.mean(list(aps.values()))

    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(title + f"\nMacro-AP={ap_macro:.3f} | Micro-AP={ap_micro:.3f}")
    ax.legend(loc="lower left")
    fig.tight_layout()
    fig.savefig(savepath, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return aps, ap_macro, ap_micro

Clase para definir el comportamiento del aprendizaje

In [32]:
from torchmetrics.classification import Accuracy, F1Score, MulticlassConfusionMatrix
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR

class LitVid(L.LightningModule):
    def __init__(self, model,
                 alpha_cb=None,              # tensor [C] o None
                 loss_type='focal',             # 'ce' (recomendado en tu setup) o 'focal'
                 class_weights=None,
                 gamma=2.0,                  # para focal
                 label_smoothing=0.05):      # sirve tanto en CE como en Focal
        super().__init__()
        self.save_hyperparameters(ignore=["model"])
        self.model = model
        self._epoch_train_start = None
        self._epoch_val_start   = None

        self.criterion = make_loss(
            loss_type=loss_type,
            alpha=alpha_cb,
            gamma=gamma,
            label_smoothing=label_smoothing,
            reduction='mean'
        )
        
        cw = torch.ones(int(NUM_CLASSES), dtype=torch.float32) if class_weights is None else class_weights.float()
        self.register_buffer("class_weights", cw, persistent=True)
        self.train_acc = Accuracy(task="multiclass", num_classes=int(NUM_CLASSES))
        self.val_acc   = Accuracy(task="multiclass", num_classes=int(NUM_CLASSES))
        self.val_f1    = F1Score(task="multiclass", num_classes=int(NUM_CLASSES), average="macro")
        self.val_cm = MulticlassConfusionMatrix(num_classes=int(NUM_CLASSES))
        self.cm_dir = CONF_MAT_DIR                 # cambia segun el num de exp
        self.roc_dir = ROC_DIR
        self.pr_dir  = PR_DIR
        self.metrics_csv = f"ROC_PR_AUC_{EXP_TAG}.csv"
        self._val_scores  = []   # para acumular probs/logits por batch
        self._val_targets = []   # para acumular labels verdaderas

        
    def forward(self, x):
        return self.model(x)

    def _step(self, batch):
        x, y = batch
        logits = self.model(x)
        loss = self.criterion(logits, y)
        preds = logits.argmax(1)
        return loss, preds, y, logits

        
    def training_step(self, batch, batch_idx):
        loss, preds, y, logits = self._step(batch)
        self.train_acc.update(preds, y)
        # Escritura de las metricas en un csv
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train_acc",  self.train_acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss
        
    def on_train_epoch_start(self):
        torch.cuda.synchronize()           # asegura tiempos precisos con GPU
        self._epoch_train_start = time.time()
        
    def on_train_epoch_end(self):
        self.train_acc.reset()
        torch.cuda.synchronize()
        train_time = time.time() - self._epoch_train_start
        print(f"⏱️ Tiempo de entrenamiento (época {self.trainer.current_epoch}): {train_time:.2f} s")
        self.log("epoch_train_time_s", train_time, prog_bar=False)

    def validation_step(self, batch, batch_idx):
        loss, preds, y, logits = self._step(batch)
        self.val_acc.update(preds, y)
        self.val_f1.update(preds, y)
        # Actualiza confusion matrix con las predicciones de este batch
        self.val_cm.update(preds, y)
        
        # === acumular scores y labels para ROC/PR ===
        probs = torch.softmax(logits, dim=1).detach().cpu()
        self._val_scores.append(probs)
        self._val_targets.append(y.detach().cpu())

        
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=False)
        return loss

    def on_validation_epoch_start(self):
        torch.cuda.synchronize()
        self._epoch_val_start = time.time()
        self._val_scores.clear()
        self._val_targets.clear()
        os.makedirs(self.cm_dir, exist_ok=True)
        os.makedirs(self.roc_dir, exist_ok=True)
        os.makedirs(self.pr_dir,  exist_ok=True)

    def on_validation_epoch_end(self):
        torch.cuda.synchronize()
        val_time = time.time() - self._epoch_val_start
        print(f"⏱️ Tiempo de validación (época {self.current_epoch}): {val_time:.2f} s")
        self.log("epoch_val_time_s", val_time, prog_bar=False)
        
        val_acc = self.val_acc.compute()
        val_f1  = self.val_f1.compute()
        self.log("val_acc", val_acc, on_epoch=True, prog_bar=True)
        self.log("val_f1",  val_f1,  on_epoch=True, prog_bar=True)
        def _tofloat(x):
            '''Utilidad para convertir a float seguro'''
            try:
                return float(x)
            except Exception:
                try:
                    return float(x.item())
                except Exception:
                    return float("vacio")
        t_loss = _tofloat(self.trainer.callback_metrics.get("train_loss", float("nan")))
        v_loss = _tofloat(self.trainer.callback_metrics.get("val_loss", float("nan")))
        t_acc  = _tofloat(self.trainer.callback_metrics.get("train_acc", float("nan")))
        v_acc  = _tofloat(val_acc)
        v_f1   = _tofloat(val_f1)
        epoch_idx = int(self.trainer.current_epoch) if hasattr(self.trainer, "current_epoch") else -1
        try:
            cm = self.val_cm.compute().detach().cpu().numpy().astype(float)  # shape [C,C]
        except Exception as e:
            print(f"[WARN] No se pudo obtener la CM: {e}")
            cm = None
        # Reset acumuladores de validación para la siguiente época
        self.val_acc.reset()
        self.val_f1.reset()
        self.val_cm.reset()

        per_class_logs = {}
        if cm is not None:
            # tp: diagonal; fp: columna - diagonal; fn: fila - diagonal
            tp = np.diag(cm)
            fp = cm.sum(axis=0) - tp
            fn = cm.sum(axis=1) - tp
            # Evitar divisiones por cero
            eps = 1e-12
            precision = (tp + eps) / (tp + fp + eps)
            recall    = (tp + eps) / (tp + fn + eps)
            f1_pc     = (2*precision*recall + eps) / (precision + recall + eps)

            for i, cname in enumerate(CLASSES):
                per_class_logs[f"val_precision/{cname}"] = float(precision[i])
                per_class_logs[f"val_recall/{cname}"]    = float(recall[i])
                per_class_logs[f"val_f1/{cname}"]        = float(f1_pc[i])

            # También se loguean promedios (por conveniencia):
            per_class_logs["val_precision_macro"] = float(np.mean(precision))
            per_class_logs["val_recall_macro"]    = float(np.mean(recall))
            per_class_logs["val_f1_macro_from_pc"] = float(np.mean(f1_pc))  # (complementa a self.val_f1)

            # Log total en CSV por época
            self.log_dict(per_class_logs, on_step=False, on_epoch=True, prog_bar=False)

            # Guardado de la matriz de confusión como imagen
            try:
                os.makedirs(self.cm_dir, exist_ok=True)
                png_raw  = os.path.join(self.cm_dir, f"confmatrix_epoch{epoch_idx:03d}_r3d18_exp2.svg")
                plot_confusion_matrix(
                    cm, CLASSES,
                    title=f"Confusion Matrix - epoch {epoch_idx} - R3D18",
                    savepath=png_raw
                )
            except Exception as e:
                print(f"[WARN] No se pudo generar/guardar la confusion matrix: {e}")

        print(f"[Epoch {epoch_idx}] train_loss={t_loss:.4f} | train_acc={t_acc:.4f} | "f"val_loss={v_loss:.4f} | val_acc={v_acc:.4f} | val_f1={v_f1:.4f}")

        # Mostrar resumen por clase de Precision / Recall / F1
        if cm is not None and per_class_logs:
            try:
                print("\n[Per-class metrics]")
                print("-" * 55)
                for cname in CLASSES:
                    p = per_class_logs.get(f"val_precision/{cname}", float("nan"))
                    r = per_class_logs.get(f"val_recall/{cname}", float("nan"))
                    f = per_class_logs.get(f"val_f1/{cname}", float("nan"))
                    print(f"  {cname:<12}  Precision: {p:6.3f} | Recall: {r:6.3f} | F1: {f:6.3f}")
                print("-" * 55)
            except Exception as e:
                print(f"[WARN] No se pudo imprimir métricas por clase: {e}")

                # === ROC/PR por época ===
        try:
            if len(self._val_scores) > 0:
                scores = torch.cat(self._val_scores, dim=0).numpy()   # [N, C]
                y_true = torch.cat(self._val_targets, dim=0).numpy()  # [N]
                epoch_idx = int(self.trainer.current_epoch)
        
                # ROC
                roc_path = os.path.join(self.roc_dir, f"roc_epoch{epoch_idx:03d}_r3d18_{EXP_TAG.lower()}.svg")
                aucs, macro_auc, micro_auc = plot_roc_ovr(
                    scores, y_true, CLASSES,
                    title=f"ROC - epoch {epoch_idx} - R3D18 ({EXP_TAG})",
                    savepath=roc_path
                )
        
                # PR
                pr_path = os.path.join(self.pr_dir, f"pr_epoch{epoch_idx:03d}_r3d18_{EXP_TAG.lower()}.svg")
                aps, macro_ap, micro_ap = plot_pr_ovr(
                    scores, y_true, CLASSES,
                    title=f"Precision-Recall - epoch {epoch_idx} - R3D18 ({EXP_TAG})",
                    savepath=pr_path
                )
        
                # CSV por época con métricas agregadas
                csv_path = self.metrics_csv
                write_header = not os.path.exists(csv_path)
                
                # leer tiempos desde los logs de la época (este scope no ve variables locales de otros hooks)
                train_time = _tofloat(self.trainer.callback_metrics.get("epoch_train_time_s", np.nan))
                val_time   = _tofloat(self.trainer.callback_metrics.get("epoch_val_time_s", np.nan))

                with open(csv_path, "a", newline="") as f:
                    import csv
                    writer = csv.writer(f)
                    if write_header:
                        header = (["epoch", "train_time_per_epoch","val_time_per_epoch"] +
                                  [f"AUC_{c}" for c in CLASSES] + ["AUC_macro", "AUC_micro"] +
                                  [f"AP_{c}"  for c in CLASSES] + ["AP_macro", "AP_micro"])
                        writer.writerow(header)
                    row = ([epoch_idx, train_time, val_time] +
                           [aucs[c] for c in CLASSES] + [macro_auc, micro_auc] +
                           [aps[c]  for c in CLASSES] + [macro_ap, micro_ap])
                    writer.writerow(row)
        except Exception as e:
            print(f"[WARN] ROC/PR no generadas: {e}")


    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=LR, weight_decay=WD)
        total_steps = int(MAX_EPOCHS * len(train_loader))
        warmup_steps = max(1, int(0.08 * total_steps)) # antes era 0.05
        cosine_steps = max(1, total_steps - warmup_steps)
        sch1 = LinearLR(opt, start_factor=0.1, total_iters=warmup_steps)
        sch2 = CosineAnnealingLR(opt, T_max=cosine_steps)
        sched = SequentialLR(opt, schedulers=[sch1, sch2], milestones=[warmup_steps])
        return {
            "optimizer": opt,
            "lr_scheduler": {
                "scheduler": sched,
                "interval": "step",   # Importante: scheduler por *pasos*
            },
        }

In [33]:
from lightning.pytorch.callbacks import Callback

class CSVLoggerCallback(Callback):
    def __init__(self, filename=f"training_metrics_{EXP_TAG.lower()}.csv"):
        super().__init__()
        self.filename = filename
        self.header_written = False
    def on_train_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        row = [
            trainer.current_epoch,
            float(metrics.get("train_loss", float("nan"))),
            float(metrics.get("train_acc", float("nan"))),
            float(metrics.get("val_loss", float("nan"))),
            float(metrics.get("val_acc", float("nan"))),
            float(metrics.get("val_f1", float("nan")))
        ]
        with open(self.filename, "a", newline="") as f:
            writer = csv.writer(f)
            if not self.header_written:
                writer.writerow(["epoch", "train_loss", "train_acc", "val_loss", "val_acc", "val_f1"])
                self.header_written = True
            writer.writerow(row)

In [34]:
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

model = torchvision.models.video.r3d_18(weights="KINETICS400_V1")
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

learning_behaviour = LitVid(model,  alpha_cb=alpha_cb, loss_type='focal', gamma=2.0, label_smoothing=0.05)

early_stop = EarlyStopping(
    monitor="val_f1",   # o "val_loss"
    mode="max",         # "min" si vigilo val_loss
    patience=10,         # nº de épocas sin mejora antes de parar
    min_delta=1e-4,     # mejora mínima para contar como “mejor”
    check_on_train_epoch_end=False
)

ckpt_saver = ModelCheckpoint(
    dirpath=f"CHECKPOINTS_{EXP_TAG}",
    monitor="val_f1",
    mode="max",
    save_top_k=1,
    save_last=True,
    filename="{epoch:02d}-{val_f1:.4f}_"+EXP_TAG.lower(),
)

trainer = L.Trainer(
    accelerator="auto",
    devices=1,
    precision="bf16" if torch.cuda.is_available() else 32,
    num_sanity_val_steps=0, # evita validación previa al entrenamiento
    max_epochs=MAX_EPOCHS,
    check_val_every_n_epoch=1,
    logger=False,
    callbacks=[CSVLoggerCallback(), early_stop, ckpt_saver],
)

/usr/local/lib/python3.11/dist-packages/lightning/fabric/connector.py:571: `precision=bf16` is supported for historical reasons but its usage is discouraged. Please set your precision to bf16-mixed instead!
INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs


## **Ciclo de entrenamiento**

In [ ]:
trainer.fit(learning_behaviour, train_loader, val_loader) # ESTE ENTRENAMIENTO PUEDE DURAR 4 HORAS
print("BEST CHECKPOINT (highest F1):", ckpt_saver.best_model_path, ckpt_saver.best_model_score)

/usr/local/lib/python3.11/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /kaggle/working/CHECKPOINTS_EXP3 exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name      | Type                      | Params | Mode 
----------------------------------------------------------------
0 | model     | VideoResNet               | 33.2 M | train
1 | criterion | FocalLoss                 | 0      | train
2 | train_acc | MulticlassAccuracy        | 0      | train
3 | val_acc   | MulticlassAccuracy        | 0      | train
4 | val_f1    | MulticlassF1Score         | 0      | train
5 | val_cm    | MulticlassConfusionMatrix | 0      | train
----------------------------------------------------------------
33.2 M    Trainable params
0         Non-trainable params
33.2 M    Total params
132.671   Total estimated model params size (MB)
97        Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

⏱️ Tiempo de validación (época 0): 55.82 s
[Epoch 0] train_loss=nan | train_acc=nan | val_loss=0.4426 | val_acc=0.5290 | val_f1=0.4860

[Per-class metrics]
-------------------------------------------------------
  Normal        Precision:  0.833 | Recall:  0.558 | F1:  0.669
  Abuse         Precision:  0.230 | Recall:  0.375 | F1:  0.285
  Fighting      Precision:  0.438 | Recall:  0.595 | F1:  0.504
-------------------------------------------------------
⏱️ Tiempo de entrenamiento (época 0): 432.43 s


Validation: |          | 0/? [00:00<?, ?it/s]

...

⏱️ Tiempo de validación (época 10): 53.13 s
[Epoch 10] train_loss=0.0247 | train_acc=0.9808 | val_loss=0.6046 | val_acc=0.5710 | val_f1=0.4986

[Per-class metrics]
-------------------------------------------------------
  Normal        Precision:  0.835 | Recall:  0.632 | F1:  0.719
  Abuse         Precision:  0.376 | Recall:  0.280 | F1:  0.321
  Fighting      Precision:  0.343 | Recall:  0.680 | F1:  0.456
-------------------------------------------------------
⏱️ Tiempo de entrenamiento (época 10): 428.66 s


Validation: |          | 0/? [00:00<?, ?it/s]

⏱️ Tiempo de validación (época 11): 53.38 s
[Epoch 11] train_loss=0.0191 | train_acc=0.9895 | val_loss=0.6036 | val_acc=0.5470 | val_f1=0.4950

[Per-class metrics]
-------------------------------------------------------
  Normal        Precision:  0.852 | Recall:  0.587 | F1:  0.695
  Abuse         Precision:  0.298 | Recall:  0.435 | F1:  0.354
  Fighting      Precision:  0.366 | Recall:  0.540 | F1:  0.436
-------------------------------------------------------
⏱️ Tiempo de entrenamiento (época 11): 429.12 s


Validation: |          | 0/? [00:00<?, ?it/s]

⏱️ Tiempo de validación (época 12): 53.52 s
[Epoch 12] train_loss=0.0146 | train_acc=0.9942 | val_loss=0.6227 | val_acc=0.5370 | val_f1=0.4724

[Per-class metrics]
-------------------------------------------------------
  Normal        Precision:  0.843 | Recall:  0.573 | F1:  0.683
  Abuse         Precision:  0.310 | Recall:  0.260 | F1:  0.283
  Fighting      Precision:  0.333 | Recall:  0.705 | F1:  0.452
-------------------------------------------------------
⏱️ Tiempo de entrenamiento (época 12): 429.53 s


# **INFERENCIA** #

# Pequena validacion de umbrales

In [ ]:
def tune_class_thresholds(val_loader, model, num_classes, device, save_path="best_thresholds.json"):
    model.eval().to(device)
    ys = []; probs = []
    with torch.no_grad():
        for x,y in val_loader:
            x = x.to(device); y = y.to(device)
            p = torch.softmax(model(x), dim=1)
            ys.append(y.cpu().numpy()); probs.append(p.cpu().numpy())
    y_true = np.concatenate(ys,0); P = np.concatenate(probs,0)
    best_f1=-1; best_tau=np.full(num_classes, 0.5, dtype=float)
    grid = [0.3,0.4,0.5,0.6,0.7]
    for tau in itertools.product(*([grid]*num_classes)):
        tau=np.array(tau); pred=[]
        for row in P:
            mask = row >= tau
            pred.append(int(np.argmax(row*mask if mask.any() else row)))
        f1 = f1_score(y_true, np.array(pred), average="macro")
        if f1 > best_f1:
            best_f1, best_tau = f1, tau
    with open(save_path,"w") as f:
        json.dump({"thresholds": best_tau.tolist(), "f1_macro": float(best_f1)}, f, indent=2)
    return best_tau, best_f1

In [ ]:
mdl = LitVid.load_from_checkpoint(ckpt_saver.best_model_path, model=model)

In [ ]:
# Calibración de umbrales por clase (robusta)
if 'val_loader' in globals():
    # 1) rehacer un loader temporal sin workers para ver errores reales
    _val_ds = val_loader.dataset
    _val_cal_loader = DataLoader(
        _val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,                      # 👈 clave para depurar
        pin_memory=False,
        collate_fn=collate_and_normalize
    )

    # 2) usar el modelo de inferencia (o tu LightningModule si su forward devuelve logits)
    _model_for_tuning = mdl if 'mdl' in globals() else learning_behaviour
    _model_for_tuning.eval().to(DEVICE)

    # 3) correr la calibración
    best_tau, best_f1 = tune_class_thresholds(
        _val_cal_loader,
        _model_for_tuning,
        num_classes=NUM_CLASSES,
        device=DEVICE
    )

    # 4) publicar thresholds en numpy (ver celda 2 abajo)
    if isinstance(best_tau, torch.Tensor):
        THRESHOLDS = best_tau.detach().cpu().numpy()
    else:
        THRESHOLDS = np.asarray(best_tau, dtype=float)

    print("Umbrales por clase:",
          dict(zip(CLASSES, [round(float(t), 3) for t in THRESHOLDS.tolist()])))
    print("F1 Macro (validación) con umbrales:", round(float(best_f1), 4))
else:
    print("[WARN] No se pudo ejecutar calibración de umbrales: falta 'val_loader'.")

In [ ]:
# Publicar THRESHOLDS global a partir de tu calibración (tolerante)
if 'best_tau' in globals():
    if isinstance(best_tau, torch.Tensor):
        THRESHOLDS = best_tau.detach().cpu().numpy()
    else:
        THRESHOLDS = np.asarray(best_tau, dtype=float)
elif 'THRESHOLDS' not in globals():
    THRESHOLDS = np.array([0.5]*NUM_CLASSES, dtype=float)  # fallback

print("THRESHOLDS:", dict(zip(CLASSES, [round(float(t), 3) for t in THRESHOLDS.tolist()])))

## **INFERENCIA NUEVOS METODOS**

In [ ]:
def pick_label_with_thresholds(prob_vec, thresholds, classes):
    """
    prob_vec: np.array [C]
    thresholds: np.array [C]
    Regla: si existe clase != 'Normal' con p_c >= tau_c, devuelve la mejor de esas.
           Si ninguna cumple, devuelve 'Normal' (si existe) o top-1 global.
    """
    try:
        normal_idx = classes.index("Normal")
    except ValueError:
        normal_idx = None

    C = len(prob_vec)

    if normal_idx is not None:
        mask = np.ones(C, dtype=bool); mask[normal_idx] = False
        candidates = np.where((prob_vec >= thresholds) & mask)[0]
        if len(candidates) > 0:
            # elegir la candidata con mayor prob
            c_star = int(candidates[np.argmax(prob_vec[candidates])])
            return c_star
        # ninguna supera su tau -> Normal
        return normal_idx
    else:
        # sin 'Normal': top-1
        return int(np.argmax(prob_vec))

In [ ]:
@torch.no_grad()
def score_video_logits_r3d(path, clip_len=CLIP_LEN, hop=CLIP_LEN//2):
    """Devuelve logits promediados sobre todos los clips del video."""
    vr = VideoReader(path, ctx=cpu(0))
    T = len(vr)
    idxs_list = []
    start = 0
    while start + clip_len <= T:
        idxs_list.append(list(range(start, start + clip_len)))
        start += hop
    if not idxs_list:
        idxs_list = [list(range(0, min(clip_len, T)))]

    mdl.eval().to(device)
    logits_all = []
    for idxs in idxs_list:
        # [T,H,W,3] -> [3,T,H,W] -> [1,3,T,H,W]
        frames = vr.get_batch(idxs).permute(3,0,1,2).float() / 255.0
        frames = frames.unsqueeze(0)
        Tcur = frames.shape[2]  # mantener T
        # resize 3D manteniendo T
        frames = F.interpolate(frames, size=(Tcur, IMG_SIZE, IMG_SIZE),
                               mode="trilinear", align_corners=False)
        # normalización con broadcasting: [1,3,1,1,1]
        frames = (frames - KINETICS_MEAN.view(1,3,1,1,1)) / KINETICS_STD.view(1,3,1,1,1)
        out = mdl(frames.to(device))  # (1,C)
        logits_all.append(out.squeeze(0).cpu())
    return torch.stack(logits_all, 0).mean(0)  # (C,)

def predict_video_label_r3d(path):
    logits = score_video_logits_r3d(path, clip_len=CLIP_LEN, hop=CLIP_LEN//2)
    prob = torch.softmax(logits, dim=0).cpu().numpy()
    c_star = pick_label_with_thresholds(prob, THRESHOLDS, CLASSES)
    return CLASSES[c_star], [float(p) for p in prob]

In [ ]:
# Demo en 3 vídeos del split de validación
for p, ci in val_items[:3]:
    lab, prob = predict_video_label_r3d(p)
    print(Path(p).name, "->", lab, [round(x,4) for x in prob])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device for inference:", device)

In [ ]:
@torch.no_grad()
def temporal_localization_r3d(path, hop=CLIP_LEN//2, min_seg_clips=2):
    """
    Devuelve spans por clase != 'Normal' usando umbral por-clase (THRESHOLDS[c]).
    """
    vr = VideoReader(path, ctx=cpu(0))
    T = len(vr)
    idxs_list = []
    start = 0
    while start + CLIP_LEN <= T:
        idxs_list.append(list(range(start, start + CLIP_LEN)))
        start += hop
    if not idxs_list:
        idxs_list = [list(range(0, min(CLIP_LEN, T)))]

    mdl.eval().to(device)
    probs = []
    for idxs in idxs_list:
        frames = vr.get_batch(idxs).permute(3,0,1,2).float() / 255.0   # [3,T,H,W]
        frames = frames.unsqueeze(0)                                    # [1,3,T,H,W]
        Tcur = frames.shape[2]
        frames = F.interpolate(frames, size=(Tcur, IMG_SIZE, IMG_SIZE),
                               mode="trilinear", align_corners=False)
        frames = (frames - KINETICS_MEAN.view(1,3,1,1,1)) / KINETICS_STD.view(1,3,1,1,1)
        logits = mdl(frames.to(device)).squeeze(0).cpu()                # (C,)
        probs.append(torch.softmax(logits, dim=0).numpy())

    probs = np.stack(probs, 0)  # [Nclips, C]

    segs = []
    for c in range(NUM_CLASSES):
        if CLASSES[c] == "Normal":
            continue
        th_c = float(THRESHOLDS[c]) if 'THRESHOLDS' in globals() else 0.5
        mask = probs[:, c] >= th_c
        s = None; best = 0.0
        for i, m in enumerate(mask):
            if m and s is None:
                s = i; best = probs[i, c]
            elif m:
                best = max(best, probs[i, c])
            elif s is not None:
                if i - s >= min_seg_clips:
                    segs.append((s, i, c, float(best)))
                s = None; best = 0.0
        if s is not None and len(mask) - s >= min_seg_clips:
            segs.append((s, len(mask), c, float(best)))

    # convertir a segundos
    fps = float(vr.get_avg_fps() or 25.0)
    out = []
    for s, e, c, b in segs:
        t0 = s * (hop / fps)
        t1 = e * (hop / fps)
        out.append({"t0_sec": round(t0, 2), "t1_sec": round(t1, 2), "class": CLASSES[c], "score": round(b, 3)})
    return out

In [ ]:
@torch.no_grad()
def _clip_probs_r3d(video_path, clip_len, hop, img_size, mean, std, classes):
    vr = VideoReader(video_path, ctx=cpu(0))
    T = len(vr)
    if T == 0:
        return 25.0, np.zeros((0, len(classes))), 0

    # ventanas
    idxs_list = []
    start = 0
    while start + clip_len <= T:
        idxs_list.append(list(range(start, start + clip_len)))
        start += hop
    if not idxs_list:
        idxs_list = [list(range(0, min(clip_len, T)))]
    N = len(idxs_list)

    mdl.eval().to(device)
    probs = []
    for idxs in idxs_list:
        frames = vr.get_batch(idxs).permute(3,0,1,2).float() / 255.0  # [3,T,H,W]
        frames = frames.unsqueeze(0)                                   # [1,3,T,H,W]
        Tcur = frames.shape[2]
        frames = F.interpolate(frames, size=(Tcur, img_size, img_size),
                               mode="trilinear", align_corners=False)
        frames = (frames - mean.view(1,3,1,1,1)) / std.view(1,3,1,1,1)
        logits = mdl(frames.to(device)).squeeze(0).cpu()               # (C,)
        p = torch.softmax(logits, dim=0).numpy()
        probs.append(p)

    fps = float(vr.get_avg_fps() or 25.0)
    probs = np.stack(probs, 0)  # [N,C]
    return fps, probs, N

def _merge_label_spans(labels, scores, hop_seconds):
    spans = []
    if len(labels) == 0:
        return spans
    s = 0
    for i in range(1, len(labels) + 1):
        if i == len(labels) or labels[i] != labels[s]:
            t0 = s * hop_seconds
            t1 = i * hop_seconds
            best = float(np.max(scores[s:i])) if i > s else float(scores[s])
            spans.append((s, i, labels[s], best, t0, t1))
            s = i
    return spans

def annotate_full_video_portable(video_path, out_path="exp1_annot_full.mp4",
                                 band_height=60, fontsize=34, include_prob=True):
    """
    Etiqueta cada tramo usando los umbrales por clase:
    - Si alguna clase != 'Normal' supera su τ_c, usa la mejor de ellas.
    - Si ninguna supera, etiquetar 'Normal' (si existe) o top-1 global.
    """
    hop = CLIP_LEN // 2
    fps, probs, N = _clip_probs_r3d(video_path, CLIP_LEN, hop, IMG_SIZE, KINETICS_MEAN, KINETICS_STD, CLASSES)
    if N == 0:
        print("No se pudieron generar clips.")
        return []

    # etiquetar con τ por clase
    labels, top_scores = [], []
    for i in range(N):
        p = probs[i]
        c_star = pick_label_with_thresholds(p, THRESHOLDS, CLASSES)
        labels.append(c_star)
        top_scores.append(float(p[c_star]))

    hop_seconds = hop / fps
    merged = _merge_label_spans(labels, top_scores, hop_seconds)

    # render (igual que tu versión, omitido por brevedad, conserva tu implementación)
    # ... (usa tu render de bandas + texto, no cambia la lógica de etiquetas) ...

    # devolver spans
    spans = [{"t0_sec": round(t0,2), "t1_sec": round(t1,2),
              "class": CLASSES[lbl_idx], "score": round(best_score,3)}
             for (_s,_e,lbl_idx,best_score,t0,t1) in merged]
    return spans

In [ ]:
path = val_items[0][0]  # primer video de validación
segments = temporal_localization_r3d(path)
print("Segmentos detectados:")
for s in segments:
    print(s)

# **INFERENCIA POR SEGMENTOS**

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from decord import VideoReader, cpu

@torch.no_grad()
def _build_windows_for_range(T, fps, t0_sec, t1_sec, clip_len, hop):
    """Devuelve lista de ventanas de índices [start..start+clip_len) para [t0,t1)."""
    # Pasar a índices (frames)
    s_idx = max(0, int(np.floor(t0_sec * fps)))
    e_idx = min(T, int(np.ceil(t1_sec * fps)))
    if e_idx <= s_idx:
        return []

    idxs_list = []
    start = s_idx
    while start + clip_len <= e_idx:
        idxs_list.append(list(range(start, start + clip_len)))
        start += hop
    if not idxs_list:  # segmento muy corto
        idxs_list = [list(range(s_idx, min(s_idx + clip_len, e_idx)))]
    return idxs_list

@torch.no_grad()
def _clip_probs_for_windows(vr, idxs_list, img_size, mean, std, model, device):
    """Inferencia por lista de ventanas de frames; retorna matriz [Nclips, C]."""
    probs = []
    model.eval().to(device)
    for idxs in idxs_list:
        frames = vr.get_batch(idxs).permute(3,0,1,2).float() / 255.0  # [3,T,H,W]
        frames = frames.unsqueeze(0)                                   # [1,3,T,H,W]
        Tcur = frames.shape[2]  # mantener T
        frames = F.interpolate(frames, size=(Tcur, img_size, img_size),
                               mode="trilinear", align_corners=False)
        frames = (frames - mean.view(1,3,1,1,1)) / std.view(1,3,1,1,1)
        logits = model(frames.to(device)).squeeze(0).cpu()             # (C,)
        probs.append(torch.softmax(logits, dim=0).numpy())
    return np.stack(probs, 0)  # [N, C]

def pick_label_with_thresholds(prob_vec, thresholds, classes):
    """Si alguna clase != 'Normal' supera su tau, devuelve la mejor; si no, 'Normal' o top-1."""
    try:
        normal_idx = classes.index("Normal")
    except ValueError:
        normal_idx = None
    C = len(prob_vec)
    if normal_idx is not None:
        mask = np.ones(C, dtype=bool); mask[normal_idx] = False
        candidates = np.where((prob_vec >= thresholds) & mask)[0]
        if len(candidates) > 0:
            return int(candidates[np.argmax(prob_vec[candidates])])
        return normal_idx
    else:
        return int(np.argmax(prob_vec))

@torch.no_grad()
def detect_segments_with_thresholds(video_path, segments,  # lista de (t0_sec, t1_sec)
                                    clip_len=CLIP_LEN, hop=None,
                                    img_size=IMG_SIZE, mean=KINETICS_MEAN, std=KINETICS_STD,
                                    model=None, device=None, classes=None, thresholds=None):
    """
    Para cada segmento [t0,t1) hace sliding-window y devuelve:
      - 'segment_pred': etiqueta elegida con umbrales
      - 'segment_prob': prob de esa etiqueta
      - 'per_clip': lista con dicts {t0_sec, t1_sec, prob_vec(list)}
    """
    if hop is None:
        hop = clip_len // 2
    if model is None:
        model = mdl
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if classes is None:
        classes = CLASSES
    if thresholds is None:
        thresholds = THRESHOLDS

    vr = VideoReader(video_path, ctx=cpu(0))
    T = len(vr)
    fps = float(vr.get_avg_fps() or 25.0)

    results = []
    for (t0, t1) in segments:
        idxs_list = _build_windows_for_range(T, fps, float(t0), float(t1), clip_len, hop)
        if not idxs_list:
            results.append({
                "t0_sec": float(t0), "t1_sec": float(t1),
                "segment_pred": None, "segment_prob": None,
                "per_clip": []
            })
            continue

        probs = _clip_probs_for_windows(vr, idxs_list, img_size, mean, std, model, device)  # [N,C]

        # tiempos por clip (cada ventana avanza 'hop' frames)
        hop_sec = hop / fps
        per_clip = []
        for i in range(len(idxs_list)):
            cprobs = probs[i]
            t0c = float(t0) + i * hop_sec
            t1c = min(float(t1), t0c + hop_sec)
            per_clip.append({"t0_sec": round(t0c, 3), "t1_sec": round(t1c, 3),
                             "probs": [float(x) for x in cprobs]})

        # agregación por segmento (promedio de clips)
        seg_prob = probs.mean(axis=0)  # [C]
        c_star = pick_label_with_thresholds(seg_prob, thresholds, classes)

        results.append({
            "t0_sec": float(t0), "t1_sec": float(t1),
            "segment_pred": classes[c_star],
            "segment_prob": float(seg_prob[c_star]),
            "per_clip": per_clip
        })

    return results

In [ ]:
# ejemplo: dividir el video en tramos de 8 segundos
def tile_segments(total_sec, seg_len=8.0):
    s=0.0; out=[]
    while s < total_sec:
        out.append((s, min(total_sec, s+seg_len)))
        s += seg_len
    return out

vr_tmp = VideoReader(val_items[0][0], ctx=cpu(0))
total_sec = len(vr_tmp) / float(vr_tmp.get_avg_fps() or 25.0)
segs = tile_segments(total_sec, seg_len=8.0)

res = detect_segments_with_thresholds(val_items[0][0], segs)
for r in res[:3]:
    print(r["t0_sec"], r["t1_sec"], "->", r["segment_pred"], round(r["segment_prob"], 3))

In [ ]:
from moviepy.editor import VideoFileClip, ImageClip, ColorClip, CompositeVideoClip
from PIL import Image, ImageDraw, ImageFont
import textwrap

def _text_image_rgba(w, h, text, fontsize=34, fg=(255,255,255), bg=(0,0,0,0)):
    img = Image.new("RGBA", (w, h), bg)
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("DejaVuSans.ttf", fontsize)
    except Exception:
        font = ImageFont.load_default()
    wrapped = textwrap.fill(text, width=max(10, int(w/18)))
    bbox = draw.multiline_textbbox((0,0), wrapped, font=font, align="center")
    tw, th = bbox[2]-bbox[0], bbox[3]-bbox[1]
    x = max(0, (w - tw)//2)
    y = max(0, (h - th)//2)
    draw.multiline_text((x, y), wrapped, font=font, fill=fg+(255,), align="center", spacing=4)
    return img

def _color_for(idx, classes):
    try:
        normal_idx = classes.index("Normal")
    except ValueError:
        normal_idx = None
    palette = [
        (128,128,128), (220, 20, 60), (255,140,0), (34,139,34), (30,144,255),
        (138,43,226), (205,92,92), (47,79,79), (70,130,180), (199,21,133)
    ]
    if normal_idx is not None and idx == normal_idx:
        return (80,80,80)
    return palette[idx % len(palette)]

def annotate_segments_video(video_path, segments, out_path="annot_segments.mp4",
                            band_height=60, fontsize=34, include_clip_probs=False):
    """
    Anota SOLO los tramos indicados. En cada tramo:
      - etiqueta del tramo según umbrales (promedio de clips),
      - probabilidad del tramo,
      - opcionalmente imprime probs por clip (resumen).
    """
    # 1) detectar por segmentos
    results = detect_segments_with_thresholds(video_path, segments)

    # 2) abrir video base
    base = VideoFileClip(video_path).without_audio()
    W, H = base.w, base.h

    # 3) crear overlays
    overlays = []
    for r in results:
        t0, t1 = r["t0_sec"], r["t1_sec"]
        label = r["segment_pred"]
        if label is None:
            continue
        c_idx = CLASSES.index(label)
        p_seg = r["segment_prob"]

        # Línea principal del tramo
        txt = f"{label}  (p={p_seg:.2f})"
        band = (ColorClip(size=(W, band_height), color=_color_for(c_idx, CLASSES))
                .set_opacity(0.35)
                .set_position(("center", H - band_height))
                .set_start(t0).set_duration(max(0.05, t1 - t0)))

        # Texto principal
        pil_img = _text_image_rgba(W, band_height, txt, fontsize=fontsize, fg=(255,255,255), bg=(0,0,0,0))
        txt_clip = (ImageClip(np.array(pil_img).astype("uint8"), transparent=True)
                    .set_position(("center", H - band_height))
                    .set_start(t0).set_duration(max(0.05, t1 - t0)))

        overlays += [band, txt_clip]

        # (opcional) mini-texto con probs por clip dentro del tramo
        if include_clip_probs and r["per_clip"]:
            # construimos una línea corta con top-1 por clip y su prob (recortada)
            mini_chunks = []
            for pc in r["per_clip"]:
                pv = np.array(pc["probs"])
                cls_i = int(np.argmax(pv))
                mini_chunks.append(f"{CLASSES[cls_i]}:{pv[cls_i]:.2f}")
            tiny_text = " | ".join(mini_chunks[:10])  # no inundar
            pil_mini = _text_image_rgba(W, int(band_height*0.65), tiny_text, fontsize=max(16, int(fontsize*0.6)),
                                        fg=(255,255,255), bg=(0,0,0,0))
            mini_clip = (ImageClip(np.array(pil_mini).astype("uint8"), transparent=True)
                        .set_position(("center", H - band_height - int(band_height*0.7)))
                        .set_start(t0).set_duration(max(0.05, t1 - t0)))
            overlays.append(mini_clip)

    # 4) componer y exportar
    comp = CompositeVideoClip([base, *overlays])
    comp.write_videofile(out_path, codec="libx264", audio=False, fps=base.fps,
                         verbose=False, logger=None)
    print("Guardado:", out_path)
    return results

In [ ]:
# define los segmentos manualmente (segundos) o con un tiler
segments = [(0, 8.0), (8.0, 16.0), (16.0, 24.0)]  # ejemplo manual

# o automáticamente:
vr_tmp = VideoReader(val_items[0][0], ctx=cpu(0))
total_sec = len(vr_tmp) / float(vr_tmp.get_avg_fps() or 25.0)
segments = [(s, min(total_sec, s+8.0)) for s in np.arange(0, total_sec, 8.0)]

i = 19
res = annotate_segments_video(
    video_path=val_items[i][0],
    segments=segments,
    out_path=f"{i}annot_segments_demo.mp4",
    include_clip_probs=True  # pon False si no quieres la línea de probs por clip
)

# 'res' tiene por cada tramo: label, prob segmento, y per-clip probs con tiempos
for r in res[:3]:
    print(r["t0_sec"], r["t1_sec"], "->", r["segment_pred"], round(r["segment_prob"], 3))